# FraudGuard: explore the completed experiment

Run from the project root or this notebook directory after installing the project. This notebook reads the measured run; it does not select a new model using the final test. See `docs/WALKTHROUGH.md` for the complete design.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if not (root / "artifacts").exists():
    root = root.parent
run = root / "artifacts" / "benchmark"
report = json.loads((run / "evaluation.json").read_text())

## Temporal windows
The five windows have distinct jobs. Only the train window fits candidate models; the test window is reporting-only.

In [ ]:
pd.DataFrame(report["splits"]).T

## Candidate selection
Compare tuning-window average precision. The highest value determines the champion, regardless of model complexity.

In [ ]:
pd.DataFrame(report["experiments"])[["name", "tune_average_precision", "seconds"]].sort_values(
    "tune_average_precision", ascending=False
)

## Final test outcomes
The threshold was fixed on the policy window. The final test did not participate in model or threshold selection.

In [ ]:
report["test"]

In [ ]:
display(Image(filename=str(run / "evaluation.png")))

## Inspect the review policy
Costs are illustrative units and assume that every reviewed fraud is caught. A validation review budget is not a guaranteed future rate limit.

In [ ]:
report["policy"]

## Score the public example with the trusted artifact
Joblib deserialization is only appropriate for trusted local releases.

In [ ]:
from fraudguard.artifacts import load_bundle
from fraudguard.data import FEATURES

model, manifest = load_bundle(run)
payload = json.loads((run / "example_request.json").read_text())
X = pd.DataFrame(payload["transactions"])[FEATURES]
probability = float(model.predict_proba(X)[0, 1])
{
    "probability": probability,
    "decision": "review" if probability >= manifest["policy"]["threshold"] else "pass",
    "model_version": manifest["run_id"],
}

## Extensions that require fresh data
Add multi-month rolling validation, point-in-time customer aggregates, label maturation, review-queue integration, and business-approved acceptance gates. Do not optimize against this already observed final test and describe the result as independent validation.